In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import torch

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from pathlib import Path

kaggle = Path('/kaggle/input/digit-recognizer')
csv_file = kaggle / "train.csv"
test_file = kaggle / "test.csv"

from torch.utils.data import Dataset

class MNISTDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)


        self.labels = torch.tensor(self.data["label"].values, dtype=torch.long)
        self.images = self.data.drop("label", axis=1).values

        # Normalize to 0-1
        self.images = self.images.astype("float32") / 255.0

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Reshape to 28x28 (28x28 represents the original image size)
        image = self.images[idx].reshape(28, 28)
        image = torch.tensor(image, dtype=torch.float).unsqueeze(0) # adds dimension representing # color channels. 28x28 -> 1x28x28 for grayscale
    
        label = self.labels[idx]
        
        return image, label



In [ ]:
from torch.utils.data import random_split, DataLoader

dataset = MNISTDataset(csv_file)

# train/val 8:2 split
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
plt.imshow(train_dataset[0][0][0,:,:])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 32, kernel_size=3, padding=1)  
    
        self.dropout = nn.Dropout(0.3) 
        self.pool = nn.MaxPool2d(2, 2)  

        self.fc = nn.Linear(32*7*7, 10)
   
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)              
        x = F.relu(self.conv2(x))
        x = self.pool(x)             
        x = F.relu(self.conv3(x))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)    # flatten
        x = self.fc(x)
        return x

In [ ]:
import torch.optim as optim

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    

In [ ]:
class MNISTTestDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file).values.astype("float32") / 255.0 

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx].reshape(28, 28)
        img = torch.tensor(img, dtype=torch.float).unsqueeze(0) 
        return img

test_dataset = MNISTTestDataset(test_file)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model.eval()
predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        predictions.extend(predicted.cpu().numpy())

submission = pd.DataFrame({
    "ImageId": list(range(1, len(predictions) + 1)),
    "Label": predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv")

In [ ]:
# Visualization
i = 0
for images in test_loader:
    image1 = images[i]
    images = images.to(device)
    outputs = model(images)
    _, predicted = outputs.max(1)
    break
plt.imshow(image1[0,:,:])
print(predicted[i].item())